In [1]:
import os
import joblib
import warnings
os.environ["OMP_NUM_THREADS"] = "1"
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.svm import SVR
from google.colab import files
import matplotlib.pyplot as plt
from keras.optimizers import Adam
from keras.models import Sequential
from keras.callbacks import EarlyStopping
from keras.layers import LSTM, Dense, Dropout
from statsmodels.tsa.arima.model import ARIMA
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Open a file upload dialog
uploaded = files.upload()

Saving Cleaned Dataset.csv to Cleaned Dataset.csv


In [4]:
dataset = pd.read_csv("Cleaned Dataset.csv")

In [5]:
# Define the target
target = 'Log_BEV Percentage (Total Number Of Registrations)'

In [6]:
# Load the PCA tools
tools = joblib.load("pca_processors.pkl")
scaler = tools['scaler']
pca = tools['pca_model']
features = tools['feature_names']

/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.3.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator PCA from version 1.3.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [7]:
# Data splitting into Pre and Post COVID
pre_covid_df = dataset[dataset['Year'] < 2020].copy()
post_covid_df = dataset[dataset['Year'] >= 2020].copy()

Please refer the jupyter notebook - **3.1.Feature Selection_BEV Percentage (Primary Target)** to understand the imputation

In [8]:
# Check which columns have NaNs and how many
print("Missing Values Per Column (Pre-COVID)")
print(pre_covid_df[features].isna().sum())

print("\nMissing Values Per Column (Post-COVID)")
print(post_covid_df[features].isna().sum())

Missing Values Per Column (Pre-COVID)
Log_Recharging Points             0
Real Range                        0
Log_Available                     0
YJ_Purchase price (EUR)           0
AC Recharging Speed (km/h)        0
Battery Capacity                  0
Log_DC Recharging Speed (km/h)    0
EG                                0
Log_GDP                           0
Log_CPI                           8
dtype: int64

Missing Values Per Column (Post-COVID)
Log_Recharging Points             0
Real Range                        0
Log_Available                     0
YJ_Purchase price (EUR)           0
AC Recharging Speed (km/h)        0
Battery Capacity                  0
Log_DC Recharging Speed (km/h)    0
EG                                0
Log_GDP                           0
Log_CPI                           1
dtype: int64


In [9]:
# Checking which rows are empty
print("Missing Log_CPI in Pre-COVID")
print(pre_covid_df[pre_covid_df['Log_CPI'].isna()].reset_index()[['Country', 'Year', 'Log_CPI']])
print("\n")
print("Missing Log_CPI in Post-COVID")
print(post_covid_df[post_covid_df['Log_CPI'].isna()].reset_index()[['Country', 'Year', 'Log_CPI']])

Missing Log_CPI in Pre-COVID
    Country  Year  Log_CPI
0  Bulgaria  2014      NaN
1   Croatia  2016      NaN
2    Cyprus  2014      NaN
3    Cyprus  2015      NaN
4    Cyprus  2016      NaN
5    Greece  2014      NaN
6    Greece  2015      NaN
7   Romania  2016      NaN


Missing Log_CPI in Post-COVID
  Country  Year  Log_CPI
0  Greece  2020      NaN


In [10]:
# Imputation for Log_CPI NaNs by filling the spaces with the median

# Calculate the median Log_CPI for every country of Pre-COVID data
country_medians = pre_covid_df.groupby('Country')['Log_CPI'].median()

# Function to fill the gaps
def impute_by_country(df):
    df_clean = df.copy()

    # Loop through the data, if Log_CPI is missing, we look up that specific country's median from our 'country_medians' list.
    df_clean['Log_CPI'] = df_clean['Log_CPI'].fillna(df_clean['Country'].map(country_medians))

    return df_clean

# Apply the fix to both datasets
pre_covid_fixed = impute_by_country(pre_covid_df)
post_covid_fixed = impute_by_country(post_covid_df)

# Extract 10 features
X_train_imputed = pre_covid_fixed[features]
X_test_imputed = post_covid_fixed[features]

In [11]:
# Scale the features using the scaler loaded from the .pkl file
X_train_scaled = scaler.transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

In [12]:
# Transform the features into 3 Principal Components using the loaded PCA model
# and convert them back to a dataframe for easy handling
X_train_pca = pd.DataFrame(pca.transform(X_train_scaled), columns=['PC1', 'PC2', 'PC3'], index=pre_covid_fixed.index)
X_test_pca = pd.DataFrame(pca.transform(X_test_scaled), columns=['PC1', 'PC2', 'PC3'], index=post_covid_fixed.index)

In [13]:
# Define the target
y_train = pre_covid_fixed[target]
y_test = post_covid_fixed[target]

print(f"Train Shape (PCs): {X_train_pca.shape}")
print(f"Test Shape (PCs): {X_test_pca.shape}")

Train Shape (PCs): (243, 3)
Test Shape (PCs): (135, 3)


In [14]:
# Define the clusters based on the K-Means clusters + PCA
leaders_list = [
    'Austria', 'Belgium', 'Czech Republic', 'Denmark', 'Finland', 'France',
    'Germany', 'Hungary', 'Italy', 'Netherlands', 'Poland',
    'Romania', 'Spain', 'Sweden'
]

followers_list = [
    'Bulgaria', 'Croatia', 'Cyprus', 'Estonia', 'Greece', 'Ireland',
    'Latvia', 'Lithuania', 'Luxembourg', 'Malta', 'Portugal', 'Slovakia', 'Slovenia'
]

In [15]:
# Filter Training Data (Pre-COVID)
X_train_leaders = X_train_pca[pre_covid_fixed['Country'].isin(leaders_list)]
y_train_leaders = y_train[pre_covid_fixed['Country'].isin(leaders_list)]

X_train_followers = X_train_pca[pre_covid_fixed['Country'].isin(followers_list)]
y_train_followers = y_train[pre_covid_fixed['Country'].isin(followers_list)]

# Filter Testing Data (Post-COVID)
X_test_leaders = X_test_pca[post_covid_fixed['Country'].isin(leaders_list)]
y_test_leaders = y_test[post_covid_fixed['Country'].isin(leaders_list)]

X_test_followers = X_test_pca[post_covid_fixed['Country'].isin(followers_list)]
y_test_followers = y_test[post_covid_fixed['Country'].isin(followers_list)]

## Neural Network Models

**1. LSTM Model**

In [16]:
def run_lstm_model(X_train, y_train, X_test, y_test, cluster_name):
    # Scaling - Crucial for Neural Networks
    scaler_X = MinMaxScaler()
    scaler_y = MinMaxScaler()

    # Reshape y to 2D for the scaler
    y_train_arr = y_train.values.reshape(-1, 1)
    y_test_arr = y_test.values.reshape(-1, 1)

    X_train_scaled = scaler_X.fit_transform(X_train)
    X_test_scaled = scaler_X.transform(X_test)
    y_train_scaled = scaler_y.fit_transform(y_train_arr)

    # Reshape to 3D [samples, time steps, features]
    X_train_3D = X_train_scaled.reshape((X_train_scaled.shape[0], 1, X_train_scaled.shape[1]))
    X_test_3D = X_test_scaled.reshape((X_test_scaled.shape[0], 1, X_test_scaled.shape[1]))

    # Build a more robust model for small data
    model = Sequential([
        LSTM(32, activation='tanh', input_shape=(1, X_train.shape[1]), return_sequences=False),
        Dropout(0.1),
        Dense(16, activation='relu'),
        Dense(1)
    ])

    # Lower learning rate (0.001) helps the model not "jump over" the solution
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse')

    # Early Stopping to prevent overfitting
    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True
    )

    # Fit the model
    model.fit(
        X_train_3D, y_train_scaled,
        epochs=200,
        batch_size=2, # Small batch size for small datasets
        validation_split=0.2,
        callbacks=[early_stop],
        verbose=0
    )

    # Predict and Inverse Scale
    test_pred_scaled = model.predict(X_test_3D)
    test_pred_log = scaler_y.inverse_transform(test_pred_scaled).flatten()

    # Back-transform from Log to Real %
    y_test_real = np.exp(y_test)
    test_pred_real = np.exp(test_pred_log)

    print(f"{cluster_name} - Improved LSTM Ready.")
    return {
        'y_test_real': y_test_real,
        'pred_test_real': test_pred_real
    }

In [17]:
# Run for both clusters
results_lstm_leaders = run_lstm_model(X_train_leaders, y_train_leaders, X_test_leaders, y_test_leaders, "Leaders")
results_lstm_followers = run_lstm_model(X_train_followers, y_train_followers, X_test_followers, y_test_followers, "Followers")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step
Leaders - Improved LSTM Ready.


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
Followers - Improved LSTM Ready.


In [18]:
# Combine predictions and actual values
combined_preds = np.concatenate([
    results_lstm_leaders['pred_test_real'],
    results_lstm_followers['pred_test_real']
])

combined_actuals = pd.concat([
    results_lstm_leaders['y_test_real'],
    results_lstm_followers['y_test_real']
])

# Combine metadata in the same order
combined_metadata = pd.concat([
    post_covid_fixed[post_covid_fixed['Country'].isin(leaders_list)],
    post_covid_fixed[post_covid_fixed['Country'].isin(followers_list)]
])

# Build final results table
all_lstm_results = pd.DataFrame({
    'Country': combined_metadata['Country'].values,
    'Year': combined_metadata['Year'].values,
    'Actual_BEV_%': combined_actuals.values,
    'Predicted_BEV_%': combined_preds
})

# Absolute error calculation
all_lstm_results['Error_Abs'] = (all_lstm_results['Actual_BEV_%'] - all_lstm_results['Predicted_BEV_%']).abs()

# Print results by year and cluster
for year in sorted(all_lstm_results['Year'].unique()):
    print(f"\n---> LSTM: {int(year)} PREDICTION RESULTS")

    year_data = all_lstm_results[all_lstm_results['Year'] == year]

    # Leaders Display
    leaders_data = (
        year_data[year_data['Country'].isin(leaders_list)]
        .sort_values('Actual_BEV_%', ascending=False)
    )
    print(f"\n**** LEADERS CLUSTER ({int(year)}) ****")
    print(leaders_data[['Country', 'Actual_BEV_%', 'Predicted_BEV_%', 'Error_Abs']].to_string(index=False))

    # Followers Display
    followers_data = (
        year_data[year_data['Country'].isin(followers_list)]
        .sort_values('Actual_BEV_%', ascending=False)
    )
    print(f"\n**** FOLLOWERS CLUSTER ({int(year)}) ****")
    print(followers_data[['Country', 'Actual_BEV_%', 'Predicted_BEV_%', 'Error_Abs']].to_string(index=False))


---> LSTM: 2020 PREDICTION RESULTS

**** LEADERS CLUSTER (2020) ****
       Country  Actual_BEV_%  Predicted_BEV_%  Error_Abs
   Netherlands         21.43         2.055252  19.374748
        Sweden         10.55         1.613916   8.936084
       Denmark          8.16         1.475672   6.684328
        France          7.72         1.154458   6.565542
       Germany          7.56         1.703726   5.856274
       Austria          7.41         1.559368   5.850632
       Finland          5.40         1.318474   4.081526
       Belgium          4.48         1.710031   2.769969
         Italy          3.35         1.384073   1.965927
       Hungary          3.32         1.661836   1.658164
       Romania          3.25         1.474792   1.775208
         Spain          3.14         1.198126   1.941874
Czech Republic          2.61         1.570830   1.039170
        Poland          1.86         1.970257   0.110257

**** FOLLOWERS CLUSTER (2020) ****
   Country  Actual_BEV_%  Predicted_BEV

In [19]:
# Save the results in an excel file

# Filter the results as per the clusters
all_lstm_results['Cluster'] = all_lstm_results['Country'].apply(
    lambda x: 'Leaders' if x in leaders_list else 'Followers'
)

# Create the Leaders and Followers dataFrame and sort in alphabetical order for easier visualisation
leaders_df = (
    all_lstm_results[all_lstm_results['Cluster'] == 'Leaders']
    .sort_values(by=['Country', 'Year'])
)

followers_df = (
    all_lstm_results[all_lstm_results['Cluster'] == 'Followers']
    .sort_values(by=['Country', 'Year'])
)

# Save the data frame in an excel file
leaders_df.to_excel("Trial1_LSTM_Leaders_Results.xlsx", index=False)
followers_df.to_excel("Trial1_LSTM_Followers_Results.xlsx", index=False)